In [1]:
## Install Libraries
%pip install pandas numpy scikit-learn matplotlib seaborn joblib streamlit

In [2]:
## Import Necessary libraries
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
## Load the dataset
df = pd.read_csv("bank.csv", sep=",")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (11162, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


In [4]:
## Data preprocessing

# Check missing values
print("Missing values:")
print(df.isnull().sum())

# Check duplicate records
print("Duplicate records:", df.duplicated().sum())

# Separate input features and target
X = df.drop("deposit", axis=1)

# Convert target into numerical values
y = df["deposit"].map({"no": 0, "yes": 1})

print("\nTarget distribution:")
print(y.value_counts())

print("\nInput features:", X.shape[1])

Missing values:
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
deposit      0
dtype: int64
Duplicate records: 0

Target distribution:
deposit
0    5873
1    5289
Name: count, dtype: int64

Input features: 16


In [5]:
## Identify numerical and categorical columns

numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

Numerical features: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Categorical features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


In [6]:
## Train-test split (80:20)

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (8929, 16)
Testing data: (2233, 16)


In [7]:
## Preprocessing pipeline

# Handle compatibility with different sklearn versions
try:
    encoder = OneHotEncoder(handle_unknown="ignore",sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore",sparse=False)

preprocessor = ColumnTransformer(transformers=[("num", StandardScaler(), numerical_features),("cat", encoder, categorical_features)],sparse_threshold=0)

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


In [8]:
## Implement all five classification models

models = {"Logistic Regression": LogisticRegression(max_iter=2000,random_state=42),
          "Decision Tree": DecisionTreeClassifier(max_depth=10,random_state=42),
          "KNN": KNeighborsClassifier(n_neighbors=5),
          "Naive Bayes": GaussianNB(),
          "Support Vector Machine": SVC(probability=True,random_state=42)
         }

print("All five classification models created!")

All five classification models created!


In [9]:
## Train and evaluate all five models

os.makedirs("model", exist_ok=True)

results = []
trained_models = {}

for name, classifier in models.items():
    print("\nTraining:", name)
    pipeline = Pipeline([("preprocessing", preprocessor),("classifier", classifier)])
    # Train model
    pipeline.fit(X_train, y_train)
    # Predictions
    y_pred = pipeline.predict(X_test)
    # Prediction probabilities
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    # Evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)

    results.append({
        "ML Model Name": name,
        "Accuracy": accuracy,
        "AUC": auc,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "MCC": mcc
    })

    trained_models[name] = pipeline

    print(name, "completed!")

print("\nAll five models trained successfully!")


Training: Logistic Regression
Logistic Regression completed!

Training: Decision Tree
Decision Tree completed!

Training: KNN
KNN completed!

Training: Naive Bayes
Naive Bayes completed!

Training: Support Vector Machine
Support Vector Machine completed!

All five models trained successfully!


In [10]:
## Display Comparison Table

results_df = pd.DataFrame(results)

display(results_df.round(4))

,ML Model Name,Accuracy,AUC,Precision,Recall,F1 Score,MCC
0,Logistic Regression,0.8262,0.9071,0.8278,0.7996,0.8135,0.6513
1,Decision Tree,0.8168,0.8590,0.7814,0.8516,0.8150,0.6366
2,KNN,0.8173,0.8796,0.8199,0.7873,0.8033,0.6333
3,Naive Bayes,0.7201,0.8042,0.7837,0.5652,0.6568,0.4472
4,Support Vector Machine,0.8554,0.9194,0.8249,0.8819,0.8524,0.7124


In [11]:
## Save All Trained Models

model_files = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "Support Vector Machine": "svm.joblib"
}

for name, pipeline in trained_models.items():
    filename = model_files[name]
    joblib.dump(pipeline,os.path.join("model", filename))

print("All five models saved successfully!")

All five models saved successfully!


In [12]:
## Save test dataset
test_data = X_test.copy()

test_data["y"] = y_test

test_data.to_csv("test_data.csv",index=False)

results_df.to_csv("metrics.csv",index=False)

print("Test dataset saved successfully!")
print("Metrics saved successfully!")

Test dataset saved successfully!
Metrics saved successfully!


In [13]:
## Check your final outputs

print("Saved model files:")

print(os.listdir("model"))

print("\nTest dataset shape:")
print(test_data.shape)

print("\nFinal model comparison:")
display(results_df.round(4))

Saved model files:
['decision_tree.joblib', 'svm.joblib', 'naive_bayes.joblib', 'knn.joblib', 'logistic_regression.joblib']

Test dataset shape:
(2233, 17)

Final model comparison:


,ML Model Name,Accuracy,AUC,Precision,Recall,F1 Score,MCC
0,Logistic Regression,0.8262,0.9071,0.8278,0.7996,0.8135,0.6513
1,Decision Tree,0.8168,0.8590,0.7814,0.8516,0.8150,0.6366
2,KNN,0.8173,0.8796,0.8199,0.7873,0.8033,0.6333
3,Naive Bayes,0.7201,0.8042,0.7837,0.5652,0.6568,0.4472
4,Support Vector Machine,0.8554,0.9194,0.8249,0.8819,0.8524,0.7124


In [14]:
!zip -r model.zip model

  adding: model/ (stored 0%)
  adding: model/decision_tree.joblib (deflated 74%)
  adding: model/svm.joblib (deflated 91%)
  adding: model/naive_bayes.joblib (deflated 50%)
  adding: model/knn.joblib (deflated 91%)
  adding: model/logistic_regression.joblib (deflated 59%)
